# 01 — Validate one SWOT pass: preprocess → py-eddy-tracker → review

Thin driver. All logic lives in `src/swot_eddy/{preprocess,detect}.py`. Method follows
de Marez et al. (2026), *Ocean Science* (doi:10.5194/os-22-1515-2026): pad 69→128, tile
128×128 / 64-overlap, biharmonic inpaint, PET (shape_error 85), R_max & gap test on the
max-velocity (speed) contour, min radius 5 km, min amplitude 0.01 m.

Pipeline: crop to Gulf Stream box → SSHA (`ssha_filtered`) + ADT (`ssha_unedited`+`mdt`)
channels, quality_flag ≤ 3 masked then biharmonic-inpainted → 128×128 tiles → PET on SSHA
→ merge + dedup. Detections are **flagged** (not dropped) so you reject by hand.

Viz: **(a)** SSHA cm `RdBu_r` ±30; **(b)** ADT m `Spectral_r` −0.1..0.8 (reference figure).

In [ ]:
import re, datetime as dt
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from swot_eddy.config import load_config
from swot_eddy import preprocess as pp
from swot_eddy.detect import detect_pass

cfg = load_config()
path = Path(cfg["data_root"]) / cfg["example_file"]
# pass timestamp from the filename (..._YYYYMMDDThhmmss_...)
m = re.search(r"_(\d{8})T(\d{6})_", path.name)
date = dt.datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S") if m else dt.datetime(2023, 8, 14)
print(path.name, "| pass date:", date)

## Preprocess

In [ ]:
tiles, summary, clon, clat = pp.preprocess_pass(str(path), cfg)
print("summary:", summary)
print("n_tiles:", len(tiles), "| tile0 data shape:", tiles[0].data.shape,
      "| any NaN in tile0:", bool(np.isnan(tiles[0].data).any()))

## Sanity check — tile 0 (SSHA, ADT, inpaint mask)\nConfirms orientation and where the nadir gap / pad sit.

In [ ]:
t = tiles[0]
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(t.data[0] * 100, cmap="RdBu_r", vmin=-30, vmax=30, origin="lower", aspect="auto")
ax[0].set_title("tile0 SSHA (cm)")
ax[1].imshow(t.data[1], cmap="Spectral_r", vmin=-0.1, vmax=0.8, origin="lower", aspect="auto")
ax[1].set_title("tile0 ADT (m)")
ax[2].imshow(t.inpaint_mask[0], cmap="gray", origin="lower", aspect="auto")
ax[2].set_title("tile0 inpaint mask (white=filled)")
for a in ax:
    a.set_xlabel("across-track px (j)"); a.set_ylabel("along-track px (i)")
plt.tight_layout(); plt.show()

## Detect (py-eddy-tracker on SSHA)

In [ ]:
dets, dsum = detect_pass(tiles, cfg, clon, clat, date)
print("detect summary:", dsum)

## Reassemble full-pass fields (what PET saw) for plotting

In [ ]:
def stitch(tiles, n_lines):
    C, _, P = tiles[0].data.shape
    full = np.full((C, n_lines, P), np.nan)
    for t in tiles:
        h = t.data.shape[1]
        e = min(t.line_start + h, n_lines)
        full[:, t.line_start:e, :] = t.data[:, :e - t.line_start, :]
    return full

n_lines = summary["n_lines_cropped"]
full = stitch(tiles, n_lines)   # (2, n_lines, 128): channel 0 ssha, channel 1 adt
pad_left = tiles[0].pad_left
print("full field shape:", full.shape)

## Overlay: SSHA & ADT with detections

Contours are the **max-velocity (speed) contours**, like de Marez Fig. 1–2. Red =
anticyclonic, blue = cyclonic. **Dashed** = `gap_reject` (paper would drop: >40% of the
contour on gaps, or >60% if the center is in the nadir band). **×** marker = center in the
nadir gap. Dotted vertical lines mark the real-data columns (`pad_left … pad_left+69`).

In [ ]:
def overlay(ax, field2d, title, cmap, vmin, vmax, scale=1.0, label=""):
    im = ax.imshow(field2d * scale, cmap=cmap, vmin=vmin, vmax=vmax,
                   origin="lower", aspect="auto")
    ax.axvline(pad_left, color="k", lw=0.6, ls=":"); ax.axvline(pad_left + 69, color="k", lw=0.6, ls=":")
    for d in dets:
        c = "red" if d.polarity == "anticyclonic" else "blue"
        ls = "--" if d.gap_reject else "-"
        if d.contour_pix:
            cl = np.array(d.contour_pix)             # (N, 2) = (line, col)
            ax.plot(cl[:, 1], cl[:, 0], color=c, lw=1.2, ls=ls)
        mk = "x" if d.center_in_nadir else ("s" if d.gap_reject else "o")
        ax.plot(d.center_col, d.center_line, mk, color=c, ms=5, mfc="none", mew=1.4)
    ax.set_title(title); ax.set_xlabel("across-track px"); ax.set_ylabel("along-track line")
    fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02).set_label(label)

fig, ax = plt.subplots(1, 2, figsize=(12, 14))
overlay(ax[0], full[0], "(a) SSHA (ssha_filtered) + detections", "RdBu_r", -30, 30, scale=100, label="η' (cm)")
overlay(ax[1], full[1], "(b) ADT (ssha_unedited + mdt)", "Spectral_r", -0.1, 0.8, label="ADT (m)")
plt.tight_layout(); plt.show()

## Detection table — manual review surface\n`radius_km` = R_max (speed radius); `gap_rej` = paper reject flag.

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame([{
        "polarity": d.polarity, "line": d.center_line, "col": d.center_col,
        "lon": None if d.center_lon is None else round(d.center_lon, 3),
        "lat": None if d.center_lat is None else round(d.center_lat, 3),
        "radius_km": round(d.radius_km, 1), "amp_m": round(d.amplitude_m, 3),
        "gap_frac": round(d.gap_frac, 2), "gap_rej": d.gap_reject,
        "nadir": d.center_in_nadir, "tile": d.source_tile,
    } for d in dets]).sort_values("amp_m", ascending=False)
    display(df)
except ImportError:
    for d in sorted(dets, key=lambda d: -d.amplitude_m):
        print(d.polarity[:4], d.center_line, d.center_col, round(d.radius_km, 1),
              round(d.amplitude_m, 3), "REJECT" if d.gap_reject else "", "nadir" if d.center_in_nadir else "")

## Parameters (de Marez vs recall)

Detection runs on `ssha_filtered`, **no high-pass**. Knobs in `config.yaml` `pet.params`:
- **`pixel_limit_min`** — paper uses PET-standard 4; we default **3** for higher small-eddy
  recall (set 4 to match de Marez exactly).
- **`shape_error = 85`** — paper value (maximizes detections).
- `step = min_amplitude_m / 2` → amplitude floor `min_amplitude_m` (0.01 m); min radius 5 km on R_max.

Detections are flagged, not dropped — carry false positives to the hand-approval step.

In [ ]:
p = cfg["pet"]["params"]
print("shape_error:", p["shape_error"], "| min_amplitude_m:", p["min_amplitude_m"],
      "| min_radius_km:", p["min_radius_km"],
      "| pixel_limit:", (p.get("pixel_limit_min", 3), p.get("pixel_limit_max", 2000)),
      "(paper-standard = (4, 1000))")